## 1. Dataset Background

The dataset was collected from DAEWOO Steel Co. Ltd. in South Korea and published by Sathishkumar, V. E. and others. It contains energy and operating measurements from a smart small-scale steel industry. The dataset covers the period from 1 January to 31 December 2018, with measurements recorded at 15-minute intervals, resulting in 35,040 observations. Each row represents the electrical and operating measurements recorded during a specific 15-minute interval.

## 2. Dataset Structure

In [1]:
# importing necessary libraries
import pandas as pd

In [2]:
# loading dataset
df = pd.read_csv('../data/Steel_industry_data.csv')

In [3]:
# checking the number of rows and columns in the dataset
df.shape

(35040, 11)

In [4]:
# checking the names of the columns in the dataset
df.columns.to_list()

['date',
 'Usage_kWh',
 'Lagging_Current_Reactive.Power_kVarh',
 'Leading_Current_Reactive_Power_kVarh',
 'CO2(tCO2)',
 'Lagging_Current_Power_Factor',
 'Leading_Current_Power_Factor',
 'NSM',
 'WeekStatus',
 'Day_of_week',
 'Load_Type']

In [5]:
# checking the data type of each column
df.dtypes

date                                        str
Usage_kWh                               float64
Lagging_Current_Reactive.Power_kVarh    float64
Leading_Current_Reactive_Power_kVarh    float64
CO2(tCO2)                               float64
Lagging_Current_Power_Factor            float64
Leading_Current_Power_Factor            float64
NSM                                       int64
WeekStatus                                  str
Day_of_week                                 str
Load_Type                                   str
dtype: object

In [6]:
#checking for missing values
df.isna().sum()

date                                    0
Usage_kWh                               0
Lagging_Current_Reactive.Power_kVarh    0
Leading_Current_Reactive_Power_kVarh    0
CO2(tCO2)                               0
Lagging_Current_Power_Factor            0
Leading_Current_Power_Factor            0
NSM                                     0
WeekStatus                              0
Day_of_week                             0
Load_Type                               0
dtype: int64

In [7]:
# checking for duplicate rows
df.duplicated().sum() 

np.int64(0)

Findings:
The dataset contains 35,040 rows and 11 columns. The variables include temporal, numerical, and categorical information. There are no missing values or duplicated rows. The date column is initially stored as a string and will be converted to a datetime format for temporal analysis.

## 3. Understanding the Variables

| Variable | What it represents | Unit | Data Type | Initial Role | 
| :- | :-: | :-: | :-: | :-: |
| date | the date and time the observation was taken | N/A | temporal | provides chronological order, allows for investigation of temporal patterns, and for time-based forecasting |
| Usage_kWh | energy consumed in a particular 15 min interval | kWh | numerical | possible numerical target variable |
| Lagging_Current_Reactive.Power_kVarh | reactive energy associated with loads where current lags the voltage, typically associated with inductive equipment | kVarh| numerical | possible predictor of usage kwh |
| Leading_Current_Reactive_Power_kVarh | reactive energy associated with loads where current leads the voltage, typically associated with capacitive behaviour | kVarh | numerical | possible predictor of usage kwh |
| CO2(tCO2) | CO₂ emissions from energy used | tonnes | numerical | potentially derived from energy consumption - investigate before using as a predictor |
| Lagging_Current_Power_Factor | indicates how effectively apparent power is being converted into useful real power | dimensionless | numerical | possible predictor of usage kwh |
| Leading_Current_Power_Factor | indicates how effectively apparent power is being converted into useful real power | dimensionless | numerical | possible predictor of usage kwh |
| NSM | how many seconds past midnight the observation was taken | seconds | temporal | helps to indicate how energy is consumed over time in a one-day period |
| WeekStatus | whether it is weekend or weekday | N/A | categorical | used to show the difference between energy usage on weekends and weekdays |
| Day_of_week | identifies the day of the week | N/A | categorical | allows to investigate differences in energy consumption across days | 
| Load_Type  | categorical classification of the operating/load condition | N/A | categorical | used to show how different levels of load correspond with energy usage | 

Usage_kWh was identified as the target variable because the primary objective is to forecast energy consumption. It represents the energy consumed during each 15-minute interval and therefore directly corresponds to the quantity being predicted.

## 4. The Time Dimension

In [8]:
# converting the date column to a datetime object
df['date'] = pd.to_datetime(df['date'], format = '%d/%m/%Y %H:%M')
df.dtypes

date                                    datetime64[us]
Usage_kWh                                      float64
Lagging_Current_Reactive.Power_kVarh           float64
Leading_Current_Reactive_Power_kVarh           float64
CO2(tCO2)                                      float64
Lagging_Current_Power_Factor                   float64
Leading_Current_Power_Factor                   float64
NSM                                              int64
WeekStatus                                         str
Day_of_week                                        str
Load_Type                                          str
dtype: object

In [9]:
# checking for the earliest and latest timestamp
earliest = df['date'].min()
print(earliest)
latest = df['date'].max()
print(latest)

2018-01-01 00:00:00
2018-12-31 23:45:00


In [10]:
# checking the interval between date samples
df['date'].diff().unique()

<TimedeltaArray>
[NaT, '0 days 00:15:00', '-1 days +00:15:00', '1 days 00:15:00']
Length: 4, dtype: timedelta64[us]

In [11]:
# checking the unusual time intervals
df['time_diff'] = df['date'].diff()
unusual = df[df['time_diff'] != pd.Timedelta(minutes=15)]
unusual[['date', 'time_diff']]

,date,time_diff
0,2018-01-01 00:15:00,NaT
95,2018-01-01 00:00:00,-1 days +00:15:00
96,2018-01-02 00:15:00,1 days 00:15:00
191,2018-01-02 00:00:00,-1 days +00:15:00
192,2018-01-03 00:15:00,1 days 00:15:00
...,...,...
34847,2018-12-29 00:00:00,-1 days +00:15:00
34848,2018-12-30 00:15:00,1 days 00:15:00
34943,2018-12-30 00:00:00,-1 days +00:15:00
34944,2018-12-31 00:15:00,1 days 00:15:00


In [12]:
# checking if the number of observations for each day is correct (24x4 =96)
df.head(96)

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,time_diff
0,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load,NaT
1,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load,0 days 00:15:00
2,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load,0 days 00:15:00
3,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load,0 days 00:15:00
4,2018-01-01 01:15:00,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load,0 days 00:15:00
...,...,...,...,...,...,...,...,...,...,...,...,...
91,2018-01-01 23:00:00,3.53,3.74,0.0,0.0,68.64,100.0,82800,Weekday,Monday,Light_Load,0 days 00:15:00
92,2018-01-01 23:15:00,3.53,3.42,0.0,0.0,71.82,100.0,83700,Weekday,Monday,Light_Load,0 days 00:15:00
93,2018-01-01 23:30:00,3.24,2.95,0.0,0.0,73.94,100.0,84600,Weekday,Monday,Light_Load,0 days 00:15:00
94,2018-01-01 23:45:00,3.67,3.96,0.0,0.0,67.97,100.0,85500,Weekday,Monday,Light_Load,0 days 00:15:00


In [13]:
df['date'].duplicated().sum()

np.int64(0)

Findings:
The dataset contains 96 observations per day, corresponding to 15-minute intervals across 24 hours. However, inspection of the timestamps shows that the 00:00 observation is positioned after the 23:45 observation within each daily block, rather than appearing at the beginning of the day. The sequence therefore requires careful handling before chronological time-series features are created. No duplicated timestamps were found.